# Geographic Arbitrage — A Guided Walkthrough

This notebook walks through the **`geoarb`** toolkit step by step and connects
each piece of code to the maths in `paper/geographic_arbitrage.pdf`.

> ⚠️ **All data are synthetic.** This is an educational laboratory, not market
> data, and not investment advice.

**Contents**
1. Build the spatial trade network
2. Generate synthetic market data
3. Verify regional prices are *cointegrated*
4. Delivered cost & arbitrage windows
5. Spatial price equilibrium (transportation LP)
6. Back-test the strategy (hedged vs unhedged)
7. Interactive maps & dashboard

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from geoarb import (build_default_network, SyntheticDGP, DGPConfig,
                    ArbitrageEngine, SpatialEquilibrium,
                    ArbitrageSimulator, SimConfig, performance_summary)
from geoarb.metrics import mean_reversion_check, adf_pvalue
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. The spatial trade network
4 export origins (South America + US Gulf) and 4 import destinations
(China + EU). Freight scales with great-circle distance. *(Paper: Definition 1.)*

In [ ]:
locations, routes = build_default_network()
origins = [l.code for l in locations if l.role=='origin']
dests   = [l.code for l in locations if l.role=='destination']
print('Origins:', origins); print('Destinations:', dests)
pd.DataFrame([dict(lane=f'{r.origin}->{r.destination}', dist_nm=r.distance_nm,
                   base_freight=r.base_freight, tariff=r.tariff)
              for r in routes]).head(8)

## 2. Synthetic market data
Prices share a common world factor `F_t` (random walk) plus a stationary local
spread; freight & basis mean-revert; FX is a log random walk.
*(Paper: Section 5.)*

In [ ]:
dgp = SyntheticDGP(DGPConfig(n_days=750, seed=42), locations, routes)
data = dgp.generate(); prices = data['prices']
fig, ax = plt.subplots(figsize=(11,4))
for c in [c for c in prices.columns if c!='date']:
    ax.plot(prices['date'], prices[c], lw=1, label=c)
ax.set_title('Synthetic location cash prices (USD/tonne)')
ax.legend(ncol=4, fontsize=7); plt.tight_layout(); plt.show()

## 3. Are regional prices cointegrated?
Sharing `F_t` makes any spread stationary & mean-reverting.
*(Paper: Proposition 3 & Corollary 1.)*

In [ ]:
for a,b in [('BR_SANTOS','AR_ROSARIO'), ('CN_QINGDAO','NL_ROTTERDAM')]:
    cc = mean_reversion_check(prices, a, b)
    print(f"{cc['pair']:>26}: AR(1)={cc['ar1_coef']:.3f}  "
          f"half-life={cc['half_life_days']:.1f}d  reverting={cc['mean_reverting']}")
spread = prices['BR_SANTOS'] - prices['AR_ROSARIO']
plt.figure(figsize=(11,3))
plt.plot(prices['date'], spread, lw=1)
plt.axhline(spread.mean(), color='r', ls='--', lw=1, label='mean')
plt.title('Cointegrated spread: Santos - Rosario'); plt.legend()
plt.tight_layout(); plt.show()

## 4. Delivered cost & arbitrage windows
`margin = dest_price - delivered_cost`; positive = open window.
*(Paper: Definitions 2-3, Proposition 1.)*

In [ ]:
engine = ArbitrageEngine(locations, routes, financing_rate=0.06)
arb = engine.compute(data)
stats = ArbitrageEngine.window_stats(arb)
display(stats.round(2))
top = stats.index[0]; g = arb[arb['lane']==top]
plt.figure(figsize=(11,3))
plt.plot(g['date'], g['arb_margin'], lw=1); plt.axhline(0, color='k', ls='--', lw=1)
plt.fill_between(g['date'], 0, g['arb_margin'], where=g['arb_margin']>0,
                 color='green', alpha=.25, label='window open')
plt.title(f'Arbitrage margin - {top}'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Spatial price equilibrium (transportation LP)
Equilibrium flows solve a min-cost transportation LP; duals are equilibrium
location prices. *(Paper: eq. (10), Theorem 1.)*

In [ ]:
spe = SpatialEquilibrium(origins, dests)
snap_date = prices['date'].iloc[-1]; snap = arb[arb['date']==snap_date]
eq = spe.solve_from_arb(snap)
print('Snapshot:', pd.Timestamp(snap_date).date(),
      '| total margin:', round(eq['total_margin'],1))
print('Supply shadow:', dict(zip(origins, eq['supply_shadow'].round(2))))
print('Demand shadow:', dict(zip(dests, eq['demand_shadow'].round(2))))
eq['flow_df'].round(2)

## 6. Back-test: hedged vs unhedged
Selling futures cancels random-walk flat-price risk, leaving the mean-reverting
spatial/basis spread. *(Paper: Proposition 5.)*

In [ ]:
results = {}
for hedge in (False, True):
    sim = ArbitrageSimulator(SimConfig(capital=60e6, hedge_with_futures=hedge))
    out = sim.run(arb, data['futures'])
    results[hedge] = (out, performance_summary(out['equity'], out['trades'], 60e6))
comp = pd.DataFrame({('hedged' if h else 'unhedged'): r[1] for h,r in results.items()}).T
display(comp[['cagr_pct','ann_vol_pct','sharpe','max_drawdown_pct',
              'win_rate_pct','n_trades']].round(2))
plt.figure(figsize=(11,4))
for h,(out,_) in results.items():
    plt.plot(out['equity']['date'], out['equity']['equity']/1e6,
             label=('hedged' if h else 'unhedged'), lw=1.5)
plt.title('Strategy equity (USD millions)'); plt.legend(); plt.tight_layout(); plt.show()

## 7. Interactive maps & dashboard

In [ ]:
from geoarb import viz
os.makedirs('../outputs', exist_ok=True)
viz.arbitrage_map(arb, date=snap_date, locations=locations, routes=routes,
                  out_path='../outputs/arb_map.html')
viz.dashboard(data, arb, results[True][0], out_path='../outputs/dashboard.html')
viz.margin_heatmap(arb, out_path='../outputs/margin_heatmap.html')
print('Wrote outputs/arb_map.html, dashboard.html, margin_heatmap.html')

---
### Takeaways
- Cointegrated prices -> spreads mean-revert (Law of One Price).
- Most windows are **closed** most of the time: the no-arb band is wide.
- Competition + capacity (the LP) close windows a naive price scan flags.
- Hedging lowers volatility by removing flat-price risk.
- Flattering Sharpes are artefacts of an idealised, frictionless lab.